### Import Libraries

In [1]:
import pandas as pd
import joblib
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import GridSearchCV, train_test_split

### Load Preprocessed

In [2]:
X_train, X_val, y_train, y_val = joblib.load("../models/processed_train_val.joblib")
X_test = joblib.load("../models/processed_test.joblib")
preprocessor = joblib.load("../models/preprocessor.joblib")
train_df = pd.read_csv("../data/train_updated.csv")
X_raw_full = train_df.drop("RiskFlag", axis=1)
y_raw_full = train_df["RiskFlag"]


### Evaluation Function

In [3]:
def evaluate(title, y_true, y_pred, y_prob=None):
    print(f"\n======== {title} ========")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1-score :", f1_score(y_true, y_pred))
    if y_prob is not None:
        print("ROC-AUC  :", roc_auc_score(y_true, y_prob))
    print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

### BASELINE NN (FULL)

In [4]:
nn_base_full = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    alpha=0.0005,
    max_iter=300,
    random_state=42
)

nn_base_full.fit(X_train, y_train)

pred_f = nn_base_full.predict(X_val)
prob_f = nn_base_full.predict_proba(X_val)[:,1]

evaluate("NN Baseline — Full Dataset", y_val, pred_f, prob_f)

f1_nn_full_base = f1_score(y_val, pred_f)


======== NN Baseline — Full Dataset ========
Accuracy : 0.8818288623457999
Precision: 0.46199407699901285
Recall   : 0.0985055777731004
F1-score : 0.16238723108952116
ROC-AUC  : 0.7242013000798001

Confusion Matrix:
 [[35560   545]
 [ 4283   468]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.98      0.94     36105
           1       0.46      0.10      0.16      4751

    accuracy                           0.88     40856
   macro avg       0.68      0.54      0.55     40856
weighted avg       0.84      0.88      0.85     40856



### Hyperparameter Tuning

In [5]:
nn_param_grid = {
    "hidden_layer_sizes": [(64,32), (128,64), (128,64,32)],
    "alpha": [0.0001, 0.0005, 0.001],
    "learning_rate": ["constant", "adaptive"]
}

nn_grid_full = GridSearchCV(
    estimator=MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=350,
        random_state=42
    ),
    param_grid=nn_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

nn_grid_full.fit(X_train, y_train)
print("Best NN FULL params:", nn_grid_full.best_params_)

nn_best_full = nn_grid_full.best_estimator_

pred_f_tuned = nn_best_full.predict(X_val)
prob_f_tuned = nn_best_full.predict_proba(X_val)[:,1]

evaluate("NN Tuned — Full Dataset", y_val, pred_f_tuned, prob_f_tuned)
f1_nn_full_tuned = f1_score(y_val, pred_f_tuned)


Fitting 3 folds for each of 18 candidates, totalling 54 fits


/home/ayush/clg/SEM5/ML/project_1/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/ayush/clg/SEM5/ML/project_1/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/ayush/clg/SEM5/ML/project_1/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/ayush/clg/SEM5/ML/project_1/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't conv

KeyboardInterrupt: 

### 20% Prep

In [ ]:
df_20 = train_df.sample(frac=0.2, random_state=42)
X_raw_20 = df_20.drop("RiskFlag", axis=1)
y_raw_20 = df_20["RiskFlag"]
X_20 = preprocessor.transform(X_raw_20)
X_train_20, X_val_20, y_train_20, y_val_20 = train_test_split(
    X_20, y_raw_20,
    test_size=0.2,
    stratify=y_raw_20,
    random_state=42
)


### Baseline NN - 20%

In [ ]:
nn_base_20 = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    alpha=0.0005,
    max_iter=300,
    random_state=42
)

nn_base_20.fit(X_train_20, y_train_20)

pred_20 = nn_base_20.predict(X_val_20)
prob_20 = nn_base_20.predict_proba(X_val_20)[:,1]

evaluate("NN Baseline — 20% Dataset", y_val_20, pred_20, prob_20)

f1_nn_20_base = f1_score(y_val_20, pred_20)


### Hyperparameter Tuning

In [ ]:
nn_grid_20 = GridSearchCV(
    estimator=MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=350,
        random_state=42
    ),
    param_grid=nn_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

nn_grid_20.fit(X_train_20, y_train_20)
print("Best NN 20% params:", nn_grid_20.best_params_)
nn_best_20 = nn_grid_20.best_estimator_

pred_20_tuned = nn_best_20.predict(X_val_20)
prob_20_tuned = nn_best_20.predict_proba(X_val_20)[:,1]

evaluate("NN Tuned — 20%", y_val_20, pred_20_tuned, prob_20_tuned)

f1_nn_20_tuned = f1_score(y_val_20, pred_20_tuned)


### Comparison Table

In [ ]:
nn_results = pd.DataFrame({
    "Model Variant": [
        "NN Full Baseline",
        "NN Full Tuned",
        "NN 20% Baseline",
        "NN 20% Tuned"
    ],
    "F1 Score": [
        f1_nn_full_base,
        f1_nn_full_tuned,
        f1_nn_20_base,
        f1_nn_20_tuned
    ]
})
nn_results


### Generate Submissions

In [ ]:
X_full_pre = preprocessor.transform(X_raw_full)
final_nn_full = nn_best_full
final_nn_full.fit(X_full_pre, y_raw_full)
pred_nn_full = final_nn_full.predict(X_test)

sub_full = pd.DataFrame({
    "ProfileID": train_df["ProfileID"].iloc[:len(X_test)],
    "RiskFlag": pred_nn_full
})
sub_full.to_csv("../submissions/submission_nn_full.csv", index=False)
print("submission_nn_full.csv saved")

X_20_pre = preprocessor.transform(X_raw_20)

final_nn_20 = nn_best_20
final_nn_20.fit(X_20_pre, y_raw_20)
pred_nn_20 = final_nn_20.predict(X_test)

pd.DataFrame({
    "ProfileID": train_df["ProfileID"].iloc[:len(X_test)],
    "RiskFlag": pred_nn_20
}).to_csv("../submissions/submission_nn_20.csv", index=False)

print("✔ submission_nn_20.csv saved")

joblib.dump(final_nn_full, "../models/nn_best_full.joblib")
joblib.dump(final_nn_20, "../models/nn_best_20.joblib")
print("Neural Network models saved successfully")
